# Fast five-way golf-club CNN

Train a compact CNN to classify a club-head image as Driver, Wood, Hybrid, Iron, or Wedge. The notebook keeps related images from the same product together, applies identical resize and normalization to every split, and applies augmentation only to training images.

Run the cells from top to bottom. The best checkpoint is saved to `models/trained/club_type_5way_cnn.pt`. The reference set is useful for validating the workflow, but add varied club-head photos before relying on the model in production.

In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
import random
import sys

import numpy as np
from PIL import Image, ImageEnhance, ImageOps
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (parent / "requirements.txt").is_file()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from swingsight.club_cnn import (
    CHECKPOINT_FORMAT,
    DEFAULT_INPUT_SIZE,
    DEFAULT_MEAN,
    DEFAULT_STD,
    ClubCnnModel,
    classify_image,
    image_to_tensor,
)

CLASS_NAMES = ("driver", "wood", "hybrid", "iron", "wedge")
DISPLAY_NAMES = {name: name.title() for name in CLASS_NAMES}
SOURCE_DIRS = {
    "driver": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "driver",
    "wood": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "wood",
    "hybrid": PROJECT_ROOT / "assets" / "club-reference-images" / "wood" / "hybrid",
    "iron": PROJECT_ROOT / "assets" / "club-reference-images" / "iron" / "irons",
    "wedge": PROJECT_ROOT / "assets" / "club-reference-images" / "iron" / "wedges",
}
CHECKPOINT_PATH = PROJECT_ROOT / "models" / "trained" / "club_type_5way_cnn.pt"
INPUT_SIZE = DEFAULT_INPUT_SIZE
SEED = 42
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Project: {PROJECT_ROOT}")
print(f"Classes: {', '.join(DISPLAY_NAMES[name] for name in CLASS_NAMES)}")
print(f"Input: {INPUT_SIZE}x{INPUT_SIZE} | normalization mean={DEFAULT_MEAN}, std={DEFAULT_STD}")

class       train  validation  test
driver      300         150    50
wood        300         150    50
hybrid      300         150    50
iron        300         150    50
wedge       300         150    50
Total: 2500 images | train=1500 validation=750 test=250


## 1. Discover images and create a grouped split

The split is performed by product group, not by individual file. This prevents related views or downloaded variants of one club from appearing in both training and validation/test data.

In [ ]:
@dataclass(frozen=True)
class Sample:
    path: Path
    label: int
    group: str


def product_group(path: Path) -> str:
    stem = path.stem.lower()
    for marker in ("_zoom", "_image", "_img", "-zoom"):
        if marker in stem:
            stem = stem.split(marker, 1)[0]
    return stem or path.stem.lower()


all_samples: list[Sample] = []
for label, class_name in enumerate(CLASS_NAMES):
    directory = SOURCE_DIRS[class_name]
    images = sorted(
        path
        for path in directory.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if len(images) < 4:
        raise ValueError(
            f"{class_name} needs at least four images; found {len(images)} in {directory}"
        )
    all_samples.extend(Sample(path, label, product_group(path)) for path in images)


def grouped_split(label: int) -> tuple[list[Sample], list[Sample], list[Sample]]:
    grouped: dict[str, list[Sample]] = defaultdict(list)
    for sample in all_samples:
        if sample.label == label:
            grouped[sample.group].append(sample)

    groups = sorted(grouped)
    if len(groups) < 3:
        raise ValueError(
            f"{CLASS_NAMES[label]} needs at least three product groups; found {len(groups)}"
        )

    random.Random(SEED + label).shuffle(groups)
    test_count = max(1, round(len(groups) * 0.10))
    validation_count = max(1, round(len(groups) * 0.30))
    while test_count + validation_count >= len(groups):
        if validation_count > 1:
            validation_count -= 1
        else:
            test_count -= 1

    def flatten(group_names: list[str]) -> list[Sample]:
        return [sample for name in group_names for sample in grouped[name]]

    test_groups = groups[:test_count]
    validation_groups = groups[test_count:test_count + validation_count]
    train_groups = groups[test_count + validation_count:]
    return flatten(train_groups), flatten(validation_groups), flatten(test_groups)


train_samples, validation_samples, test_samples = [], [], []
for label in range(len(CLASS_NAMES)):
    train_part, validation_part, test_part = grouped_split(label)
    train_samples.extend(train_part)
    validation_samples.extend(validation_part)
    test_samples.extend(test_part)


def group_names(samples: list[Sample]) -> set[str]:
    return {f"{sample.label}:{sample.group}" for sample in samples}


train_groups = group_names(train_samples)
validation_groups = group_names(validation_samples)
test_groups = group_names(test_samples)
assert not train_groups & validation_groups
assert not train_groups & test_groups
assert not validation_groups & test_groups

print("class       train  validation  test")
for label, class_name in enumerate(CLASS_NAMES):
    counts = {
        split_name: sum(sample.label == label for sample in samples)
        for split_name, samples in (
            ("train", train_samples),
            ("validation", validation_samples),
            ("test", test_samples),
        )
    }
    print(f"{class_name:<10}{counts['train']:>5}{counts['validation']:>12}{counts['test']:>6}")
print(
    f"Total: {len(all_samples)} images | "
    f"train={len(train_samples)} validation={len(validation_samples)} test={len(test_samples)}"
)

## 2. Define preprocessing and data loaders

Every image is converted to RGB, resized to the model input size, converted to a tensor, and normalized with the checkpoint values. Training gets mild image augmentation before this shared deterministic preprocessing; validation and test do not.

In [ ]:
def augment_training_image(image: Image.Image) -> Image.Image:
    if random.random() < 0.5:
        image = ImageOps.mirror(image)
    image = image.rotate(
        random.uniform(-8, 8),
        resample=Image.Resampling.BILINEAR,
        fillcolor=(245, 245, 245),
    )
    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.90, 1.10))
    return ImageEnhance.Contrast(image).enhance(random.uniform(0.90, 1.10))


def preprocess_image(image: Image.Image) -> torch.Tensor:
    """Apply the exact resize and normalization used by validation and inference."""
    return image_to_tensor(image, INPUT_SIZE, DEFAULT_MEAN, DEFAULT_STD)


class ClubDataset(Dataset):
    def __init__(self, samples: list[Sample], training: bool) -> None:
        self.samples = samples
        self.training = training

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        sample = self.samples[index]
        with Image.open(sample.path) as opened:
            image = opened.convert("RGB")
        if self.training:
            image = augment_training_image(image)
        return preprocess_image(image), torch.tensor(sample.label, dtype=torch.long)


train_dataset = ClubDataset(train_samples, training=True)
validation_dataset = ClubDataset(validation_samples, training=False)
test_dataset = ClubDataset(test_samples, training=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Using CUDA: NVIDIA GeForce RTX 4070 SUPER | CUDA 12.8
Training 316,421 parameters on cuda.
epoch 01/35 train_loss=1.3501 validation_loss=1.4417 validation_accuracy=0.345
epoch 02/35 train_loss=1.2011 validation_loss=2.5966 validation_accuracy=0.304
epoch 03/35 train_loss=1.1709 validation_loss=3.9561 validation_accuracy=0.244
epoch 04/35 train_loss=1.1179 validation_loss=1.1798 validation_accuracy=0.411
epoch 05/35 train_loss=1.0680 validation_loss=2.6890 validation_accuracy=0.239
epoch 06/35 train_loss=1.0479 validation_loss=8.7603 validation_accuracy=0.215
epoch 07/35 train_loss=1.0285 validation_loss=4.3731 validation_accuracy=0.271
epoch 08/35 train_loss=1.0377 validation_loss=3.4269 validation_accuracy=0.291
epoch 09/35 train_loss=0.9955 validation_loss=1.0847 validation_accuracy=0.500
epoch 10/35 train_loss=0.9526 validation_loss=6.5858 validation_accuracy=0.215
epoch 11/35 train_loss=0.9486 validation_loss=1.1640 validation_accuracy=0.495
epoch 12/35 train_loss=0.9398 validation

KeyboardInterrupt: 

## 3. Verify preprocessing and labels

This quick check catches the two common pipeline failures before a CUDA run: inconsistent tensor shapes or normalization, and mismatched class labels between loaders.

In [ ]:
train_batch, train_labels = next(iter(train_loader))
validation_batch, validation_labels = next(iter(validation_loader))

assert train_batch.shape[1:] == (3, INPUT_SIZE, INPUT_SIZE)
assert validation_batch.shape == train_batch.shape
assert train_labels.min().item() >= 0
assert train_labels.max().item() < len(CLASS_NAMES)
assert validation_labels.min().item() >= 0
assert validation_labels.max().item() < len(CLASS_NAMES)

print(f"Train batch: {tuple(train_batch.shape)}")
print(f"Validation batch: {tuple(validation_batch.shape)}")
print(f"Train tensor range: {train_batch.min().item():.2f} to {train_batch.max().item():.2f}")
print(f"Validation tensor range: {validation_batch.min().item():.2f} to {validation_batch.max().item():.2f}")
print("Class mapping:")
for label, class_name in enumerate(CLASS_NAMES):
    print(f"  {label}: {class_name}")

## 4. Configure CUDA training

The lower learning rate gives validation metrics room to settle on this small dataset. Class weighting handles the modest class-count differences without changing the label mapping.

In [ ]:
EPOCHS = 35
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 8
GRADIENT_CLIP_NORM = 1.0
REQUIRE_CUDA = True


def select_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}")
        torch.backends.cudnn.benchmark = True
        return device
    if REQUIRE_CUDA:
        raise RuntimeError(
            "CUDA is required for this notebook, but this Python environment has no CUDA-enabled PyTorch build. "
            "Install the CUDA PyTorch wheel, restart the kernel, and run this cell again."
        )
    mps = getattr(torch.backends, "mps", None)
    return torch.device("mps") if mps is not None and mps.is_available() else torch.device("cpu")


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, list[int], list[int]]:
    model.eval()
    total_loss = 0.0
    total = 0
    actual, predicted = [], []

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            total_loss += float(criterion(logits, labels).item()) * labels.size(0)
            total += labels.size(0)
            actual.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(dim=1).cpu().tolist())

    accuracy = sum(a == b for a, b in zip(actual, predicted)) / max(1, total)
    return total_loss / max(1, total), accuracy, actual, predicted


device = select_device()
model = ClubCnnModel(len(CLASS_NAMES)).to(device)
class_counts = Counter(sample.label for sample in train_samples)
class_weights = torch.tensor(
    [len(train_samples) / (len(CLASS_NAMES) * class_counts[label]) for label in range(len(CLASS_NAMES))],
    dtype=torch.float32,
    device=device,
)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(f"Training {sum(parameter.numel() for parameter in model.parameters()):,} parameters on {device}.")
print(f"Learning rate: {LEARNING_RATE} | weight decay: {WEIGHT_DECAY} | label smoothing: 0.05")

## 5. Train and save the best checkpoint

Validation is measured after every epoch using the deterministic validation loader. The checkpoint is selected by validation accuracy, while early stopping limits a run that stops improving.

In [ ]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_accuracy = -1.0
stale_epochs = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_total = 0.0
    train_count = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()

        train_loss_total += float(loss.item()) * labels.size(0)
        train_count += labels.size(0)

    validation_loss, validation_accuracy, _, _ = evaluate(
        model, validation_loader, criterion, device
    )
    train_loss = train_loss_total / max(1, train_count)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            "validation_accuracy": validation_accuracy,
        }
    )
    print(
        f"epoch {epoch:02d}/{EPOCHS} "
        f"train_loss={train_loss:.4f} "
        f"validation_loss={validation_loss:.4f} "
        f"validation_accuracy={validation_accuracy:.3f}"
    )

    if validation_accuracy >= best_accuracy:
        best_accuracy = validation_accuracy
        stale_epochs = 0
        torch.save(
            {
                "format": CHECKPOINT_FORMAT,
                "task": "club_type_5way",
                "architecture": "club_cnn_small_v1",
                "class_names": list(CLASS_NAMES),
                "input_size": INPUT_SIZE,
                "mean": list(DEFAULT_MEAN),
                "std": list(DEFAULT_STD),
                "validation_accuracy": round(float(validation_accuracy), 6),
                "state_dict": {
                    name: value.detach().cpu()
                    for name, value in model.state_dict().items()
                },
            },
            CHECKPOINT_PATH,
        )
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f"Early stopping after {PATIENCE} epochs without improvement.")
            break

print(f"Best validation accuracy: {best_accuracy:.3f}")
print(f"Saved checkpoint: {CHECKPOINT_PATH}")

## 6. Evaluate the held-out test set

Only the saved best checkpoint is evaluated here. This keeps the test set separate from model selection and reports both the confusion matrix and per-class precision/recall.

In [ ]:
from time import perf_counter

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

test_loss, test_accuracy, test_actual, test_predicted = evaluate(
    model, test_loader, criterion, device
)
print(f"Held-out test loss: {test_loss:.4f}")
print(f"Held-out test accuracy: {test_accuracy:.3f}")

confusion_matrix = np.zeros((len(CLASS_NAMES), len(CLASS_NAMES)), dtype=int)
for actual, predicted in zip(test_actual, test_predicted):
    confusion_matrix[actual, predicted] += 1

print("\nConfusion matrix (rows=actual, columns=predicted):")
print("          " + " ".join(f"{name[:5]:>6}" for name in CLASS_NAMES))
for index, class_name in enumerate(CLASS_NAMES):
    values = " ".join(f"{value:>6}" for value in confusion_matrix[index])
    print(f"{class_name:<9} {values}")

print("\nPer-class precision / recall:")
for index, class_name in enumerate(CLASS_NAMES):
    true_positive = confusion_matrix[index, index]
    precision = true_positive / max(1, confusion_matrix[:, index].sum())
    recall = true_positive / max(1, confusion_matrix[index, :].sum())
    print(f"{DISPLAY_NAMES[class_name]:<8} precision={precision:.3f} recall={recall:.3f}")

checkpoint["test_accuracy"] = round(float(test_accuracy), 6)
torch.save(checkpoint, CHECKPOINT_PATH)


def predict_club_type(
    image_path: str | Path,
    minimum_confidence: float = 0.60,
) -> dict:
    with Image.open(image_path) as opened:
        result = classify_image(
            opened.convert("RGB"),
            checkpoint_path=CHECKPOINT_PATH,
            expected_task="club_type_5way",
            minimum_confidence=minimum_confidence,
        )
    return {
        "label": DISPLAY_NAMES.get(result.label, "Unknown") if result.label else "Unknown",
        "confidence": round(result.confidence, 3),
        "probabilities": {
            DISPLAY_NAMES.get(name, name): score
            for name, score in result.probabilities.items()
        },
        "reasoning": result.reasoning,
    }


example_path = test_samples[0].path
print(f"\nExample: {example_path.name}")
print(predict_club_type(example_path))

for _ in range(3):
    predict_club_type(example_path, minimum_confidence=0.0)
runs, started = 30, perf_counter()
for _ in range(runs):
    predict_club_type(example_path, minimum_confidence=0.0)
print(f"Average local inference time: {(perf_counter() - started) * 1000 / runs:.1f} ms/image")
print(f"Deployable checkpoint: {CHECKPOINT_PATH}")